In [ ]:
import pandas as pd
import pandas_gbq
import numpy as np
import warnings
import sys
import os
from google.colab import userdata
from google.cloud import bigquery

In [ ]:
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

In [ ]:
# Replace 'your_kaggle_username' with your actual Kaggle username
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY') # This will fetch the secret you added

Now, we will run a small query just to activate the BigQuery API

In [ ]:
# IMPORTANT: Replace 'your-gcp-project-id' with your actual Google Cloud Project ID
# You can create one for free at console.cloud.google.com and enable the BigQuery API there.
# This project ID is used for billing and resource management in GCP.
project_id = userdata.get('GOOGLE-CUSTOMER-ANALYTICS-PROJECT-ID')

# Define the BigQuery table path. For Google Analytics Sample, it's typically:

# Let's query a small sample from multiple tables using a wildcard for August 2017
# This will concatenate data from all ga_sessions_201708XX tables.
query = f"""
SELECT
  _TABLE_SUFFIX AS table_date,
  date,
  fullVisitorId,
  visitNumber,
  hits.page.pagePath AS pagePath,
  hits.page.pageTitle AS pageTitle
FROM
  `bigquery-public-data.google_analytics_sample.ga_sessions_201708*`,
  UNNEST(hits) AS hits
WHERE _TABLE_SUFFIX BETWEEN '01' AND '03' -- Example: query data for Aug 1st to Aug 3rd
LIMIT 10
"""

print(f"Executing BigQuery query")

# Load data into a pandas DataFrame
try:
    sample_df = pandas_gbq.read_gbq(query, project_id=project_id, dialect='standard')
    display(sample_df.head())
    print(len(sample_df))
except Exception as e:
    print(f"Error loading data from BigQuery: {e}")
    print("Please ensure you have a valid project_id and that BigQuery API is enabled for your project.")

Executing BigQuery query
Downloading: 100%|██████████|


,table_date,date,fullVisitorId,visitNumber,pagePath,pageTitle
0,01,20170801,3418334011779872055,1,/google+redesign/bags/google+zipper+front+spor...,Page Unavailable
1,01,20170801,2474397855041322408,2,/google+redesign/shop+by+brand/youtube,Page Unavailable
2,01,20170801,5870462820713110108,1,/google+redesign/shop+by+brand/youtube,Page Unavailable
3,01,20170801,9397809171349480379,1,/google+redesign/shop+by+brand/youtube,Page Unavailable
4,01,20170801,6089902943184578335,1,/google+redesign/shop+by+brand/youtube,Page Unavailable


10


In [ ]:
del sample_df

## Helpers

In [ ]:
project_id = userdata.get('GOOGLE-CUSTOMER-ANALYTICS-PROJECT-ID') # Previously defined
full_table_name = '`bigquery-public-data.google_analytics_sample.ga_sessions_*`'
client = bigquery.Client(project=project_id)

In [ ]:
def year_month_string(year, month):
  return str(year) + '-' + str(month).zfill(2)

Creating the following timestamp functions so that I can add them to the results for easy time-series analysis in visualization software

In [ ]:
def month_start_timestamp(year, month):
  return int(pd.Timestamp(year, month, 1).timestamp())

In [ ]:
def add_month_start_timestamp(df):
  if 'year' not in df.columns or 'month' not in df.columns:
    print("Input DataFrame must have 'year' and 'month' columns. Skipping Function On DataFrame")
  else:
    df['month_start_timestamp'] = df.apply(lambda row: month_start_timestamp(row['year'], row['month']), axis=1)
  return df

## Overall Summary

In [ ]:
overall_summary_query = f'''
SELECT
  COALESCE(COUNT(DISTINCT fullVisitorId), 0) AS unique_visitors,
  COALESCE(SUM(totals.transactions), 0) AS total_transactions,
  COALESCE(SUM(totals.transactionRevenue), 0) / 1000000 AS total_revenue_usd,
  COALESCE(SAFE_DIVIDE(
    COUNT(DISTINCT CASE
        WHEN totals.transactions IS NOT NULL
        THEN fullVisitorId
    END),
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS user_conversion_rate,
  COALESCE(SAFE_DIVIDE(
    COUNTIF(totals.transactions IS NOT NULL),
    COUNT(*)
  ), 0) AS session_conversion_rate,
  COALESCE(SAFE_DIVIDE(
    COALESCE(SUM(totals.transactionRevenue),0) / 1000000,
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS revenue_per_user,
  COALESCE(SAFE_DIVIDE(
    COALESCE(SUM(totals.transactionRevenue),0)/1000000,
    COALESCE(SUM(totals.transactions),0)
  ), 0) AS avg_order_value,
  COALESCE(AVG(totals.timeOnSite), 0) AS avg_time_on_site_seconds,
  COALESCE(AVG(totals.pageviews), 0) AS avg_pageviews_per_session
FROM {full_table_name}
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
'''

overall_summary_df = client.query(overall_summary_query).to_dataframe()
print(overall_summary_df)

   unique_visitors  total_transactions  total_revenue_usd  \
0           714167               12115         1540071.24   

   user_conversion_rate  session_conversion_rate  revenue_per_user  \
0              0.014033                 0.012784          2.156458   

   avg_order_value  avg_time_on_site_seconds  avg_pageviews_per_session  
0       127.121027                262.612141                   3.849764  


## Daily, Weekly, Monthly, and Overall KPIs

Creating primarily daily KPIs so that I can aggregate by week, month, and year in Tableau

In [ ]:
daily_kpis_query = f"""
SELECT
  DATE(TIMESTAMP_SECONDS(visitStartTime)) AS date,

  COALESCE(COUNT(DISTINCT fullVisitorId), 0) AS daily_active_users,

  COALESCE(SUM(totals.transactions), 0) AS daily_total_transactions,

  COALESCE(SUM(totals.transactionRevenue)/1000000, 0) AS daily_total_revenue_usd,

  COALESCE(SAFE_DIVIDE(
    COUNT(DISTINCT CASE
      WHEN totals.transactions > 0
      THEN fullVisitorId
    END),
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS daily_user_conversion_rate,

  COALESCE(SAFE_DIVIDE(
    COUNTIF(totals.transactions > 0),
    COUNT(*)
  ), 0) AS daily_session_conversion_rate,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS daily_avg_revenue_per_user,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    COUNT(*)
  ), 0) AS daily_avg_revenue_per_session,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    SUM(totals.transactions)
  ), 0) AS daily_avg_order_value

FROM {full_table_name}
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY date
ORDER BY date
"""

daily_kpis_df = client.query(daily_kpis_query).to_dataframe()
print(daily_kpis_df)

           date  daily_active_users  daily_total_transactions  \
0    2016-08-01                1197                        29   
1    2016-08-02                1799                        23   
2    2016-08-03                2481                         0   
3    2016-08-04                2959                        12   
4    2016-08-05                2745                        43   
..          ...                 ...                       ...   
362  2017-07-29                1676                        18   
363  2017-07-30                1513                        22   
364  2017-07-31                2292                        57   
365  2017-08-01                2322                        52   
366  2017-08-02                 597                         6   

     daily_total_revenue_usd  daily_user_conversion_rate  \
0                    5971.73                    0.024227   
1                    1505.52                    0.012229   
2                       0.00           

In [ ]:
weekly_kpis_query = f"""
SELECT
  DATE_TRUNC(
    DATE(TIMESTAMP_SECONDS(visitStartTime)),
    WEEK(MONDAY)
  ) AS start_of_week,

  COALESCE(COUNT(DISTINCT fullVisitorId), 0) AS weekly_active_users,

  COALESCE(SUM(totals.transactions), 0) AS weekly_total_transactions,

  COALESCE(SUM(totals.transactionRevenue)/1000000, 0) AS weekly_total_revenue_usd,

  COALESCE(SAFE_DIVIDE(
    COUNT(DISTINCT CASE
      WHEN totals.transactions > 0
      THEN fullVisitorId
    END),
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS weekly_user_conversion_rate,

  COALESCE(SAFE_DIVIDE(
    COUNTIF(totals.transactions > 0),
    COUNT(*)
  ), 0) AS weekly_session_conversion_rate,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS weekly_avg_revenue_per_user,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    COUNT(*)
  ), 0) AS weekly_avg_revenue_per_session,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    SUM(totals.transactions)
  ), 0) AS weekly_avg_order_value

FROM {full_table_name}
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY start_of_week
ORDER BY start_of_week
"""

weekly_kpis_df = client.query(weekly_kpis_query).to_dataframe()
print(weekly_kpis_df)

   start_of_week  weekly_active_users  weekly_total_transactions  \
0     2016-08-01                13476                        131   
1     2016-08-08                14773                        287   
2     2016-08-15                15030                        335   
3     2016-08-22                14182                        363   
4     2016-08-29                15136                        222   
5     2016-09-05                14013                        216   
6     2016-09-12                14486                        191   
7     2016-09-19                14629                        210   
8     2016-09-26                14185                        213   
9     2016-10-03                18350                        231   
10    2016-10-10                17137                        205   
11    2016-10-17                21552                        214   
12    2016-10-24                24221                        203   
13    2016-10-31                23160           

In [ ]:
monthly_kpis_query = f"""
SELECT
  EXTRACT(YEAR FROM TIMESTAMP_SECONDS(visitStartTime)) AS year,
  EXTRACT(MONTH FROM TIMESTAMP_SECONDS(visitStartTime)) AS month,
  DATE_TRUNC(
    DATE(TIMESTAMP_SECONDS(visitStartTime)),
    MONTH
) AS start_of_month,
  COALESCE(COUNT(DISTINCT fullVisitorId), 0) AS monthly_active_users,

  COALESCE(SUM(totals.transactions), 0) AS monthly_total_transactions,

  COALESCE(SUM(totals.transactionRevenue)/1000000, 0) AS monthly_total_revenue_usd,

  COALESCE(SAFE_DIVIDE(
    COUNT(DISTINCT CASE
      WHEN totals.transactions > 0
      THEN fullVisitorId
    END),
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS monthly_user_conversion_rate,

  COALESCE(SAFE_DIVIDE(
    COUNTIF(totals.transactions > 0),
    COUNT(*)
  ), 0) AS monthly_session_conversion_rate,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS monthly_avg_revenue_per_user,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    COUNT(*)
  ), 0) AS monthly_avg_revenue_per_session,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    SUM(totals.transactions)
  ), 0) AS monthly_avg_order_value

FROM {full_table_name}
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY year, month, start_of_month
ORDER BY year, month, start_of_month
"""

monthly_kpis_df = client.query(monthly_kpis_query).to_dataframe()
print(monthly_kpis_df)

    year  month start_of_month  monthly_active_users  \
0   2016      8     2016-08-01                 61185   
1   2016      9     2016-09-01                 59235   
2   2016     10     2016-10-01                 84536   
3   2016     11     2016-11-01                 99495   
4   2016     12     2016-12-01                 64650   
5   2017      1     2017-01-01                 52857   
6   2017      2     2017-02-01                 51331   
7   2017      3     2017-03-01                 57958   
8   2017      4     2017-04-01                 55735   
9   2017      5     2017-05-01                 52197   
10  2017      6     2017-06-01                 51890   
11  2017      7     2017-07-01                 58604   
12  2017      8     2017-08-01                  2869   

    monthly_total_transactions  monthly_total_revenue_usd  \
0                         1236                  154589.19   
1                          904                  125865.92   
2                          921  

In [ ]:
overall_kpis_query = f'''
SELECT
  COALESCE(COUNT(DISTINCT fullVisitorId), 0) AS overall_active_users,

  COALESCE(SUM(totals.transactions), 0) AS overall_total_transactions,

  COALESCE(SUM(totals.transactionRevenue)/1000000, 0) AS overall_total_revenue_usd,

  COALESCE(SAFE_DIVIDE(
    COUNT(DISTINCT CASE
      WHEN totals.transactions > 0
      THEN fullVisitorId
    END),
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS overall_user_conversion_rate,

  COALESCE(SAFE_DIVIDE(
    COUNTIF(totals.transactions > 0),
    COUNT(*)
  ), 0) AS overall_session_conversion_rate,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    COUNT(DISTINCT fullVisitorId)
  ), 0) AS overall_avg_revenue_per_user,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    COUNT(*)
  ), 0) AS overall_avg_revenue_per_session,

  COALESCE(SAFE_DIVIDE(
    SUM(totals.transactionRevenue)/1000000,
    SUM(totals.transactions)
  ), 0) AS overall_avg_order_value

FROM {full_table_name}
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
'''

overall_kpis_df = client.query(overall_kpis_query).to_dataframe()
print(overall_kpis_df)

   overall_active_users  overall_total_transactions  \
0                714167                       12115   

   overall_total_revenue_usd  overall_user_conversion_rate  \
0                 1540071.24                      0.014033   

   overall_session_conversion_rate  overall_avg_revenue_per_user  \
0                         0.012784                      2.156458   

   overall_avg_revenue_per_session  overall_avg_order_value  
0                         1.704273               127.121027  


## Acquisition

In [ ]:
acquisition_query = f"""
SELECT
  DATE(TIMESTAMP_SECONDS(visitStartTime)) AS date,

  COALESCE(geoNetwork.continent,'Unknown') AS continent,

  COALESCE(trafficSource.source,'Unknown') AS source,

  COALESCE(trafficSource.medium,'Unknown') AS medium,

  COALESCE(trafficSource.campaign,'Unknown') AS campaign,

  COALESCE(COUNT(DISTINCT fullVisitorId), 0) AS users,

  COALESCE(COUNT(*), 0) AS sessions,

  COALESCE(SUM(totals.transactions), 0) AS transactions,

  COALESCE(SUM(totals.transactionRevenue)/1000000, 0) AS revenue_usd

FROM {full_table_name}
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY
  date,
  continent,
  source,
  medium,
  campaign
ORDER BY date, continent, source, medium, campaign
"""

acquisition_df = client.query(acquisition_query).to_dataframe()
print(acquisition_df)

             date  continent         source     medium          campaign  \
0      2016-08-01  (not set)       (direct)     (none)         (not set)   
1      2016-08-01  (not set)         google    organic         (not set)   
2      2016-08-01     Africa       (direct)     (none)         (not set)   
3      2016-08-01     Africa         google    organic         (not set)   
4      2016-08-01     Africa  google.com.ng   referral         (not set)   
...           ...        ...            ...        ...               ...   
24474  2017-08-02     Europe    youtube.com   referral         (not set)   
24475  2017-08-02    Oceania       (direct)     (none)         (not set)   
24476  2017-08-02    Oceania       Partners  affiliate  Data Share Promo   
24477  2017-08-02    Oceania     google.com   referral         (not set)   
24478  2017-08-02    Oceania    youtube.com   referral         (not set)   

       users  sessions  transactions  revenue_usd  
0          1         1             

## Funnel Table

In [ ]:
funnel_query = f"""
SELECT
  CONCAT(fullVisitorId, '-', visitId) AS session_id,

  COALESCE(MAX(IF(h.type='PAGE',1,0)), 0) AS viewed,

  COALESCE(MAX(
    IF(
      h.eventInfo.eventCategory='Enhanced Ecommerce'
      AND h.eventInfo.eventAction='Add to Cart',
      1,
      0
    )
  ), 0) AS carted,

  COALESCE(MAX(IF(t.totals.transactions > 0,1,0)), 0) AS purchased

FROM {full_table_name} t,
UNNEST(t.hits) h

WHERE t._TABLE_SUFFIX BETWEEN '20160801' AND '20170801'

GROUP BY session_id
"""

funnel_df = client.query(funnel_query).to_dataframe()
print(funnel_df)

                            session_id  viewed  carted  purchased
0       1144262134219529854-1476732440       1       0          0
1        532049236738168487-1476740757       1       0          0
2       5187478915243403497-1476717197       1       0          0
3       5784539437943530054-1476693023       1       0          0
4       0396973395485053796-1476739145       1       0          0
...                                ...     ...     ...        ...
902750  2598211261955388637-1479807774       1       0          0
902751  3351834782103883977-1479862661       1       0          0
902752  6933686820770984392-1479844877       1       0          0
902753  8987908780101484296-1479865121       1       0          0
902754  2358611952979102774-1479837847       1       0          0

[902755 rows x 4 columns]


## User Retention Rate (Monthly Cohorts)

In [ ]:
# Query to get the first visit month for each unique fullVisitorId
first_visit_month_query = f'''
SELECT
  fullVisitorId,
  MIN(EXTRACT(YEAR FROM TIMESTAMP_SECONDS(visitStartTime))) AS first_year,
  MIN(EXTRACT(MONTH FROM TIMESTAMP_SECONDS(visitStartTime))) AS first_month
FROM {full_table_name}
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY fullVisitorId
'''

first_visit_month_df = client.query(first_visit_month_query).to_dataframe()

# Query all monthly visits for each fullVisitorId
monthly_visits_query = f'''
SELECT
  fullVisitorId,
  EXTRACT(YEAR FROM TIMESTAMP_SECONDS(visitStartTime)) AS visit_year,
  EXTRACT(MONTH FROM TIMESTAMP_SECONDS(visitStartTime)) AS visit_month
FROM {full_table_name}
WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
GROUP BY fullVisitorId, visit_year, visit_month
'''

monthly_visits_df = client.query(monthly_visits_query).to_dataframe()

# Merge to get cohort information for each monthly visit
retention_df = pd.merge(
    monthly_visits_df,
    first_visit_month_df,
    on='fullVisitorId',
    how='left'
)

In [ ]:
del monthly_visits_df, first_visit_month_df

In [ ]:
# Create cohort month_year string
retention_df['cohort'] = retention_df.apply(lambda row: year_month_string(row['first_year'], row['first_month']), axis=1)
retention_df['visit_month_year'] = retention_df.apply(lambda row: year_month_string(row['visit_year'], row['visit_month']), axis=1)

# Calculate the difference in months between cohort month and visit month
# First, convert 'cohort' and 'visit_month_year' to datetime objects
retention_df['cohort_dt'] = pd.to_datetime(retention_df['cohort'])
retention_df['visit_dt'] = pd.to_datetime(retention_df['visit_month_year'])

# Calculate the month difference
retention_df['month_diff'] = ((retention_df['visit_dt'].dt.year - retention_df['cohort_dt'].dt.year) * 12 + (retention_df['visit_dt'].dt.month - retention_df['cohort_dt'].dt.month))

# Count unique users per cohort and month_diff
cohort_counts = retention_df.groupby(['cohort', 'month_diff'])['fullVisitorId'].nunique().reset_index()

# Pivot the table to create a retention matrix
retention_matrix = cohort_counts.pivot_table(index='cohort', columns='month_diff', values='fullVisitorId')

# Calculate retention rate (percentage)
cohort_sizes = retention_matrix.iloc[:,0]
retention_matrix = retention_matrix.divide(cohort_sizes, axis=0)

# Filter out cohorts that have NaN values in the first month (month_diff=0) or the last cohort (August 2017) which is incomplete
# Cohorts with NaN values in month_diff=0 are those whose first visit was outside the _TABLE_SUFFIX range (Jan-Jul 2016)
retention_matrix = retention_matrix.loc[(retention_matrix[0].notna()) & (retention_matrix.index != '2017-08')]

# Won't convert to percentages just yet; will do that in viz software

# Fill remaining NaN values with 0 for display, as they represent 0% retention
display(retention_matrix.fillna(0).round(4))

month_diff,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
cohort,,,,,,,,,,,,,,,,,,,,
2016-08,1.0,0.0398,0.0154,0.0073,0.0044,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0001,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-09,1.0,0.0375,0.0115,0.0053,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-10,1.0,0.0285,0.0077,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-11,1.0,0.0241,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-12,1.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2017-01,1.0,0.0328,0.0105,0.0061,0.0044,0.0028,0.0020,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2017-02,1.0,0.0327,0.0095,0.0069,0.0034,0.0024,0.0002,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2017-03,1.0,0.0305,0.0101,0.0048,0.0034,0.0001,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2017-04,1.0,0.0341,0.0101,0.0053,0.0004,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Unpivot the retention_matrix to a long format
retention_long_df = retention_matrix.reset_index().melt(
    id_vars=['cohort'],
    var_name='month_diff',
    value_name='retention_rate'
)

# Fill NaN values with 0, as they represent 0% retention for that month
retention_long_df['retention_rate'] = retention_long_df['retention_rate'].fillna(0)

# Display the first few rows of the long format DataFrame
display(retention_long_df.head(15))

,cohort,month_diff,retention_rate
0,2016-08,0,1.000000
1,2016-09,0,1.000000
2,2016-10,0,1.000000
3,2016-11,0,1.000000
4,2016-12,0,1.000000
5,2017-01,0,1.000000
6,2017-02,0,1.000000
7,2017-03,0,1.000000
8,2017-04,0,1.000000
9,2017-05,0,1.000000


In [ ]:
print(f"Num. rows in retention_long_df: {len(retention_long_df)}")
print(f"Num. columns in retention_long_df: {len(retention_long_df.columns)}")

Num. rows in retention_long_df: 240
Num. columns in retention_long_df: 3


In [ ]:
del retention_df, retention_matrix

## User Segmentation

In [ ]:
user_segmentation_query = f"""
SELECT
  fullVisitorId,

  COALESCE(COUNT(*), 0) AS sessions,

  COALESCE(SUM(totals.transactions), 0) AS transactions,

  COALESCE(SUM(totals.transactionRevenue)/1000000, 0) AS revenue,

  MIN(DATE(TIMESTAMP_SECONDS(visitStartTime))) AS first_visit,

  MAX(DATE(TIMESTAMP_SECONDS(visitStartTime))) AS last_visit

FROM {full_table_name}

WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'

GROUP BY fullVisitorId
"""

user_segmentation_df = client.query(user_segmentation_query).to_dataframe()
print(user_segmentation_df)

              fullVisitorId  sessions  transactions  revenue first_visit  \
0       5265975655263451991         1             0     0.00  2016-08-16   
1       0824409685962808817         9             1    68.18  2016-08-16   
2       7171290027410037351         5             0     0.00  2016-08-16   
3       6059529052219252326        22             0     0.00  2016-08-05   
4       5067462012695207413         2             0     0.00  2016-08-16   
...                     ...       ...           ...      ...         ...   
714162  0092467632764575806         1             0     0.00  2016-09-06   
714163  3517574264712831072         1             0     0.00  2016-09-07   
714164  2193060275278222240         1             0     0.00  2016-09-06   
714165  7659235301227414076         1             0     0.00  2016-09-06   
714166  9850138238904592883         1             1    95.31  2016-09-06   

        last_visit  
0       2016-08-16  
1       2017-04-11  
2       2016-11-16  
3  

## Export the dataframes to CSVs for visualization software

In [ ]:
df_variables = [name for name in globals() if name.endswith('_df') and isinstance(globals()[name], pd.DataFrame)]
print("DataFrames (variables ending with '_df'):")
for var_name in df_variables:
    print(var_name)

DataFrames (variables ending with '_df'):
overall_summary_df
daily_kpis_df
weekly_kpis_df
monthly_kpis_df
overall_kpis_df
acquisition_df
funnel_df
retention_long_df
user_segmentation_df


In [ ]:
mkdir 'datafiles'

In [ ]:
# Get names of all variables ending in '_df' that are Pandas DataFrames
df_variables = [name for name in globals() if name.endswith('_df') and isinstance(globals()[name], pd.DataFrame)]

# Export each dataframe to a CSV file in the 'datafiles' directory
for df_name in df_variables:
    df = globals()[df_name]

    # For the retention_matrix, ensure NaNs are filled with 0 and the index (cohort) is preserved.
    # For other dataframes, a simple export should suffice.
    if df_name == 'retention_long_df':
        # Ensure month_diff is treated as an integer for cleaner export if it's not already
        df['month_diff'] = df['month_diff'].astype(int)
        df.to_csv(f'datafiles/{df_name}.csv', index=False)
    else:
        df.to_csv(f'datafiles/{df_name}.csv', index=False)

print("All identified dataframes have been exported to CSV files in the 'datafiles' directory.")

All identified dataframes have been exported to CSV files in the 'datafiles' directory.
